<a href="https://colab.research.google.com/github/Rita11254/notebook_python_1_1/blob/notebook_1/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22tasks_RU_import_scope_closure_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## До того как вы приступите к решению:
**Tools → Settings → Editor → completions / suggestions / linting → disable**

## Задание 1

Допишите 2 реализации функции `increment()`, которая **увеличивает глобальную переменную `counter` на 1**:

**с/без** (!) python синтаксического сахара. Сигнатуру функции менять нельзя.

**1.1: с python синтаксическим сахаром**

In [ ]:
counter = 0

def increment():
  global counter
  counter += 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter = } -- great!')

counter = 2 -- great!


**1.2: без python синтаксического сахара**

In [ ]:
counter = 0

def increment():
  globals()['counter'] += 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter = } -- great!')

counter = 2 -- great!


## Задание 2

Достаньте **только функцию `sqrt`** из модуля `math` и исполните sqrt(169).  
Нельзя исполнять `import math`.

Подготовьте 2 решения.

...

In [ ]:
import importlib

sqrt = getattr(importlib.import_module('math'), 'sqrt')
result = sqrt(169)

assert result == 13, 'try again'
print(f'{result = } -- great!')

result = 13.0 -- great!


In [ ]:
sqrt = getattr(__import__('math'), 'sqrt')
result = sqrt(169)

print(f'{result = } -- great!')

result = 13.0 -- great!


## Задание 3

Динамический импорт и перезагрузка.

1. Создайте модуль `mod.py`:

In [ ]:
# я проверяю вот так (vs code)
# есть файлик mod.py такой
msg = "A"

# файл 3.py
import mod
import importlib
for i in range(10000):
  importlib.reload(mod)
  print(mod.msg)

# в какой-то момент обновляю mod.py
msg = "B"

ModuleNotFoundError: No module named 'mod'

In [ ]:
# %%writefile mod.py
# msg = "A"

2. Импортируйте его и выведите `msg`

3. Измените `msg` на `B` в файле

4. Без перезагрузки сессии ноутбука, выведите новое значение `msg`

In [ ]:
%%writefile mod.py
msg = "A"

Writing mod.py


In [ ]:
import mod
import importlib
importlib.reload(mod)

print(mod.msg)

A


In [ ]:
%%writefile mod.py
msg = "B"

Overwriting mod.py


In [ ]:
import importlib
importlib.reload(mod)

print(mod.msg)

B


## Задание 4

У вас есть дирректория `pkg`:

In [ ]:
!mkdir -p pkg

In [ ]:
%%writefile pkg/m1.py
pi = 3.1415_92_65
_e = 2.7
__i = -1

Overwriting pkg/m1.py


```
pkg/
└── m1.py
```

Ниже ячейки для вашего кода, а после задание

### **Первый способ**: через модуль из стандартной библиотеки CPython

In [ ]:
import os
os.makedirs('pkg', exist_ok=True)

with open('pkg/__init__.py', 'w') as f:
    f.write('from .m1 import pi\n')
    f.write('__all__ = ["pi"]\n')

In [ ]:
%%writefile pkg/__init__.py
from .m1 import pi
__all__ = ['pi']

Overwriting pkg/__init__.py


### **Второй способ**: в одну строчку без доп.модулей

In [ ]:
import pkg
pi = getattr(pkg, 'pi')

### Текст задания:
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.  
Вам необходимо изменить структуру `pkg` пакета / содержимое его модулей, чтобы следующий код выполнялся корректно:

In [ ]:
from pkg import *
print(pi)

3.14159265


**Важно!**  
При обновлении любых данных в дирректории проекта, вам необходимо перезагружать сессию ipynb:  
`Runtime --> Restart Session` и перезапустить необходимые ячейки задания,  
иначе результаты могут быть для вас некорректными.

## Задание 5

При правильно решённом **задании 4** вам необходимо:
- изменить `pkg`
- дописать код ниже

так, чтобы "дотянуться" до `__i`.  
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.    
Если вы решите перезагрузить сессию, то для решения **задания 5** необходимо перезапустить ячейки **задания 4**.  

In [ ]:
# после 4ой задачи необходими удалиьт __init__

with open('pkg/__init__.py', 'w') as f:
    f.write('from .m1 import pi, _e, __i\n')
    f.write('__all__ = ["pi", "_e", "__i"]\n')

### Решение

In [ ]:
from pkg import *
print(__i)

-1


## Задание 6

Изменяемое замыкание. Почему этот код ведёт себя неожиданно? Исправьте.

In [ ]:
def create_accumulators():
    accs = []
    for i in range(3):
        def accumulator(x):
            accs[i] += x
            return accs[i]
        accs.append(0)
    return accs

acc_list = create_accumulators()
print(acc_list[0](10))  # Ожидается 10, но получается ошибка

TypeError: 'int' object is not callable

Первая ошибка, это nonlocal accs нет (не видно что хотим изменить в этом случае). Дальше посмотрим варианты. Меня интересует только первые 3 элемента из accs. Проблема в том, что функция accumulator применяется только к i = 2 (последнему), но при этом в массиве еще может не существовать 2го(начиная с 0) элемента


In [ ]:
# чтобы не было путаницы с тем, что функцию одну вызываю 3 раза, то перебор по массиву добавлю внутри функции
def create_accumulators():
    accs = [0, 0, 0]
    def accumulator(x):
        nonlocal accs
        for i in range(3):
            accs[i] += x
        return accs
    return accumulator

acc_list = create_accumulators()
print(acc_list(10)[0])

10


In [ ]:
# отдельно воздействую на каждый элемент массива (то есть есть еще замыкание inner и оно меняет конкретный элемент, а потом соединяем)
def create_accumulators():
    accs = [0, 0, 0]

    def accumulator(i):
        def inner(x):
            accs[i] += x
            return accs[i]
        return inner

    return [accumulator(i) for i in range(3)]

acc_list = create_accumulators()
print(acc_list[0](10))

10


## Задание 7
При перезапуске сессии ноутбука решение задачи начинается сначала

### 7.1: Востановите работу `print`, не используя `del`

In [ ]:
print = 1

In [ ]:
def print(arg):
  import builtins
  builtins.print(arg)

print("234")
print(1)

234
1


### 7.1: Удалите объект `print`, после востановите его функционал

In [ ]:
globals().pop('print')

from builtins import print
print("234")

234


## Задание 8

Замыкание с изменяемым состоянием. Создайте функцию-счётчик, которая запоминает количество вызовов между разными экземплярами:

In [ ]:
def make_shared_counter():
    if not('count' in globals()):
        globals()['count'] = 0
    def inner():
        globals()['count'] += 1
        return globals()['count']
    return inner

c1 = make_shared_counter()
c2 = make_shared_counter()

print(c1())
print(c2())
print(c1())

1
2
3


## Задание 9

Допишите код, чтобы функция `outer` возвращала **словарь с тремя замыканиями**: `add()`, `mul()`, `get()` — работающими с одной и той же закрытой переменной `value`.

In [ ]:
def outer(val = 0):
    count = val

    def add(x):
        nonlocal count
        count += x
        return count

    def mul(x):
        nonlocal count
        count *= x
        return count

    def get():
        nonlocal count
        return count

    return {'add' : add, 'mul' : mul, 'get' : get}

obj = outer(10)
obj['add'](5)
obj['mul'](2)
assert obj['get']() == 30, 'ну не'
print(f'{obj['get'] = } -- great!')

obj['get'] = <function outer.<locals>.get at 0x78c4b7f513a0> -- great!


## Задание 10

Создать closure, которая принимает функцию и возвращает новую функцию с кэшированием результатов (мемоизацией).

*Теоретическая справка:*

Функция `memoize` принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

В примере с функцией `fib` (числа Фибоначчи) мемоизация уменьшает количество рекурсивных вызовов с экспоненциального до линейного, так как результаты для каждого n вычисляются один раз.

Таким образом, мемоизация экономит время за счёт памяти — класическая оптимизация "время против памяти" — и особенно полезна для функций с дорогими вычислениями и повторяющимися входами.

Алгоритм для решения:

- Функция memoize принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

- Внутренняя функция wrapper проверяет, есть ли для данного входного аргумента (x) уже вычисленный результат в словаре cache.

- Если результат есть, то он возвращается из кеша, и вычисления не повторяются.

- Если нет, то вызывается исходная функция func(x), результат сохраняется в cache и возвращается.


In [ ]:
def memoize(func):
    cache = {}

    def wrapper(x):
        if x in cache:
            return cache[x]
        else:
            result = func(x)
            cache[x] = result
            return result

    return wrapper

@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

print(fib(10))

55
